In [ ]:
# Install all libs as needed
pip install praw pandas openpyxl datetime
import praw
import pandas as pd
import datetime
import time
import csv
import os
from google.colab import drive

In [ ]:
# API Info
reddit = praw.Reddit(
    client_id='Your client_id',
    client_secret='Your client_secret',
    user_agent='Your user_agent via Reddit',
    username='Your Reddit Username',
    password='Your Reddit password')

In [ ]:
# Your interested Subreddit list
sub_reddits = ['Subreddit 1', 'Subreddit 2', '....']

In [ ]:
# Create a folder to save metadata
data_folder = "Your Path"

In [ ]:
# Actual Data Collection Starts Here
if not os.path.exists(data_folder):
    os.makedirs(data_folder)

start_index = 0
end_index = len(sub_reddits)

def get_date(created_utc, timezone_offset=-5): #Added timezone offset, defaults to EST
    """Converts UTC timestamp to datetime object and applies timezone offset."""
    utc_datetime = datetime.datetime.fromtimestamp(created_utc)
    est_datetime = utc_datetime + datetime.timedelta(hours=timezone_offset)
    return est_datetime

def get_est_offset():
    """Determines the correct EST/EDT offset."""
    now = datetime.datetime.now()
    #US Daylight savings time rules.
    us_dst_start = datetime.datetime(now.year, 3, 8, 2) + datetime.timedelta(days=(6 - now.weekday()) % 7)
    us_dst_end = datetime.datetime(now.year, 11, 1, 2) + datetime.timedelta(days=(6 - now.weekday()) % 7)

    if us_dst_start <= now < us_dst_end:
        return -4  # EDT
    else:
        return -5  # EST

for i in range(start_index, min(end_index, len(sub_reddits))):
    sub_reddit = sub_reddits[i]
    data = []

    try:
        subreddit = reddit.subreddit(sub_reddit)
        print(f"Processing subreddit: {sub_reddit}")

        today_utc = int(datetime.datetime.utcnow().timestamp())
        today_midnight_utc = today_utc - (today_utc % 86400)

        est_offset = get_est_offset() #Get the correct offset.
        today_midnight_est = get_date(today_midnight_utc, est_offset).timestamp()

        for post in subreddit.new(limit=None):
            if post.created_utc >= (today_midnight_est - est_offset * 3600): #adjust the utc midnight to the est midnight.
                post_data = {
                    "Subreddit": subreddit.display_name,
                    "Post ID": post.id,
                    "Post Title": post.title,
                    "Post Text": post.selftext,
                    "Post Author": post.author.name if post.author else None,
                    "Post Created": get_date(post.created_utc, est_offset),
                }
                data.append(post_data)
            elif post.created_utc < (today_midnight_est - est_offset * 3600): #adjust the utc midnight to the est midnight.
                break

        today_date = datetime.datetime.now().strftime("%Y_%m_%d")
        csv_filename = os.path.join(data_folder, f"{today_date}_{sub_reddit}.csv")

        with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = data[0].keys() if data else []
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for row in data:
                writer.writerow(row)

        print(f"Data for {sub_reddit} exported to {csv_filename}")

        time.sleep(1)

    except Exception as e:
        print(f"Error processing subreddit {sub_reddit}: {e}")
        continue

print("Data collection completed.")